# UBKI: обробка компаній
OPF → статус → КВЕДи → частки володіння → країни → збереження

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

# ================== ШЛЯХИ ==================
INPUT_PATH = r""    # <-- шлях до ubki_all_flat.csv (або .xlsx) після ubki_flatten.py
OUTPUT_PATH = r""   # <-- куди зберегти результат (.csv); порожнє = поруч з INPUT, суфікс _processed
OPF_PATH = r"M:\Controlling\Data_Science_Projects\Financial_indicators_tool\OPFCD_OPFNM.xlsx"
# ===========================================


def move_after(df, cols, after):
    """Переставляє колонки cols одразу після колонки after."""
    rest = [c for c in df.columns if c not in cols]
    i = rest.index(after) + 1
    return df[rest[:i] + list(cols) + rest[i:]]


def norm(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.lower()
        .str.replace(r"[«»\"'“”„`’]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [ ]:
INPUT_PATH = Path(INPUT_PATH)
if INPUT_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")
else:
    df = pd.read_excel(INPUT_PATH, dtype=str).fillna("")

df["ЄДРПОУ"] = df["ЄДРПОУ"].str.strip().str.zfill(8)
print(df.shape)
df.head(3)

## 1. OPF (організаційно-правова форма)

In [ ]:
opf_df = pd.read_excel(OPF_PATH, dtype={"FIRM_OPFCD": str})
opf_ref = (
    opf_df[["FIRM_OPFCD", "FIRM_OPFNM"]]
    .dropna()
    .drop_duplicates()
    .rename(columns={"FIRM_OPFCD": "OPF_CODE", "FIRM_OPFNM": "OPF_NAME"})
)
opf_ref["OPF_CODE"] = opf_ref["OPF_CODE"].str.strip().str.replace(r"\.0$", "", regex=True)
opf_ref["OPF_NAME_NORM"] = norm(opf_ref["OPF_NAME"])
opf_ref = opf_ref[opf_ref["OPF_NAME_NORM"] != ""]

# довші назви перевіряємо першими:
# «товариство з обмеженою відповідальністю» має спрацювати раніше за «товариство»
opf_ref = (
    opf_ref.assign(_len=opf_ref["OPF_NAME_NORM"].str.len())
    .sort_values("_len", ascending=False)
    .drop_duplicates("OPF_NAME_NORM")
)

full_name_norm = norm(df["Повна назва"]) + " "
opf_code = pd.Series(pd.NA, index=df.index, dtype="object")
opf_name = pd.Series(pd.NA, index=df.index, dtype="object")

for code_, name_, name_norm in opf_ref[["OPF_CODE", "OPF_NAME", "OPF_NAME_NORM"]].itertuples(index=False):
    m = opf_code.isna() & full_name_norm.str.startswith(name_norm + " ")  # збіг цілими словами
    if m.any():
        opf_code[m] = code_
        opf_name[m] = name_

df["OPF_CODE"] = opf_code
df["OPF_NAME"] = opf_name
df = move_after(df, ["OPF_CODE", "OPF_NAME"], "Повна назва")

print(f"OPF не знайдено: {df['OPF_CODE'].isna().sum()} з {len(df)}")
# найчастіші початки назв без OPF — щоб доповнити довідник за потреби
df.loc[df["OPF_CODE"].isna(), "Повна назва"].str.split().str[:3].str.join(" ").value_counts().head(20)

In [ ]:
df["OPF_NAME"].value_counts(dropna=False).head(20)

## 2. Статус

In [ ]:
STATUS_MAP = {
    "Зареєстровано": "ACTIVE",

    "В стані припинення": "SUSPENDED",

    "Припинено": "TERMINATED",
    "Скасовано": "TERMINATED",
    "Архівний": "ARCHIVED",

    "Порушено справу про банкрутство": "BANKRUPTCY",
    "Порушено справу про банкрутство (санація)": "BANKRUPTCY",

    "Зареєстровано, свідоцтво про державну реєстрацію недійсне": "INVALID_REGISTRATION",
}

df["STATUS"] = df["Статус"].str.strip().map(STATUS_MAP).fillna("OTHER")
df = move_after(df, ["STATUS"], "Статус")

print("Статуси, що не потрапили в STATUS_MAP (OTHER):")
print(df.loc[df["STATUS"] == "OTHER", "Статус"].value_counts())
df["STATUS"].value_counts()

## 3. КВЕДи
`Код КВЕД 94.99 Опис; 93.11 Опис; ...` → `CODE_KVED_N` + `DESCR_KVED_N` (кількість пар = максимальна кількість КВЕДів у компанії)

In [ ]:
KVED_RE = re.compile(r"(\d{2}\.\d{2})\s+(.+?)(?=\s*;\s*\d{2}\.\d{2}\s|\s*;?\s*$)")


def parse_kved(s):
    s = re.sub(r"^\s*Код\s+КВЕД\s*", "", str(s).strip())
    return [(c, d.strip()) for c, d in KVED_RE.findall(s)]


kved = df["Види діяльності"].map(parse_kved)
max_kved = kved.map(len).max()
print("Макс. КВЕДів у компанії:", max_kved)

bad = (df["Види діяльності"].str.strip() != "") & (kved.map(len) == 0)
print("Не розпізнано КВЕДів у непорожніх рядках:", bad.sum())

kved_cols = {"KVED_COUNT": kved.map(len)}
for i in range(max_kved):
    kved_cols[f"CODE_KVED_{i + 1}"] = [k[i][0] if i < len(k) else "" for k in kved]
    kved_cols[f"DESCR_KVED_{i + 1}"] = [k[i][1] if i < len(k) else "" for k in kved]

pos = df.columns.get_loc("Види діяльності")
df = pd.concat(
    [df.iloc[:, :pos], pd.DataFrame(kved_cols, index=df.index), df.iloc[:, pos + 1:]],
    axis=1,
)
df.filter(regex=r"^(KVED_COUNT|CODE_KVED_[12]|DESCR_KVED_[12])$").head()

In [ ]:
df.loc[bad, ["ЄДРПОУ", "Повна назва"]].head(10)  # рядки, де КВЕДи не розпізнались

## 4. Частки володіння бенефіціарів (`Бенеф_1_Частка` … `Бенеф_20_Частка`)
Значення перезаписуються числом 0–100 прямо в цих колонках:
- текст → число (`50%`, `50,5` → `50.5`); значення поза 0–100 → порожньо
- якщо в компанії 1 бенефіціар і частка порожня → `100`
- якщо бенефіціарів ≥ 2 і порожня частка лише в одного → `100 − сума решти` (якщо результат у межах 0–100)
- записи на кшталт «КІНЦЕВИЙ БЕНЕФІЦІАРНИЙ ВЛАСНИК ВІДСУТНІЙ» не рахуються як бенефіціари
- компанії, де бенефіціарів > 20 (список обрізано), не дораховуються
- `Бенеф_ЧАСТКА_ДОРАХОВАНО = 1` там, де частку заповнено правилом

In [ ]:
PREF = "Бенеф"
SHARE_KEY = "Частка"
NAME_KEY = "ПІБ / Назва"
MIN_PERSONS_FOR_REST = 2   # з якої кількості бенефіціарів рахувати «100 − сума решти»

NOT_PERSON_RE = re.compile(r"КІНЦЕВ\w*\s+БЕНЕФІЦІАР|ВІДСУТН|НЕ\s+БАЗУЄТЬСЯ", re.I)
NUM_RE = re.compile(r"(\d+(?:[.,]\d+)?)")


def to_pct(s):
    s = str(s).strip().replace(" ", "")
    m = NUM_RE.findall(s)
    if not m:
        return np.nan
    v = float(m[-1].replace(",", "."))
    return v if 0 <= v <= 100 else np.nan


share_cols = sorted(
    [c for c in df.columns if re.fullmatch(rf"{PREF}_\d+_{SHARE_KEY}", c)],
    key=lambda c: int(c.split("_")[1]),
)
cap = len(share_cols)
print(f"Колонок з часткою: {cap}")
print("Приклади до обробки:", df.loc[df[share_cols[0]] != "", share_cols[0]].head(5).tolist())

S = np.column_stack([df[c].map(to_pct).to_numpy(float) for c in share_cols])
names = np.column_stack([df[f"{PREF}_{n}_{NAME_KEY}"].to_numpy(str) for n in range(1, cap + 1)])
is_person = (np.char.str_len(np.char.strip(names)) > 0) & ~np.vectorize(
    lambda x: bool(NOT_PERSON_RE.search(x))
)(names)

cnt = pd.to_numeric(df[f"{PREF}_кількість"], errors="coerce").fillna(0).to_numpy(int)
complete = cnt <= cap  # усі бенефіціари компанії є в колонках

missing = is_person & np.isnan(S)
n_person = is_person.sum(1)
n_missing = missing.sum(1)
known_sum = np.where(is_person & ~np.isnan(S), S, 0).sum(1)
pos_missing = missing.argmax(1)
rows = np.arange(len(df))

rule1 = complete & (n_person == 1) & (n_missing == 1)
rest = np.round(100 - known_sum, 2)
rule2 = complete & (n_person >= MIN_PERSONS_FOR_REST) & (n_missing == 1) & (rest > 0) & (rest <= 100)

S[rows[rule1], pos_missing[rule1]] = 100
S[rows[rule2], pos_missing[rule2]] = rest[rule2]

for i, c in enumerate(share_cols):
    df[c] = np.round(S[:, i], 2)   # перезапис у ту саму колонку

df[f"{PREF}_ЧАСТКА_ДОРАХОВАНО"] = (rule1 | rule2).astype(int)
df = move_after(df, [f"{PREF}_ЧАСТКА_ДОРАХОВАНО"], f"{PREF}_кількість")

print(f"1 бенефіціар → 100: {rule1.sum()}")
print(f"100 − сума решти: {rule2.sum()}")
print(f"пропущено (бенефіціарів > {cap}): {(~complete).sum()}")
print(f"сума часток > 100 у компаній: {(np.nansum(S, 1) > 100.01).sum()}")

In [ ]:
df.filter(regex=r"^Бенеф_(кількість|ЧАСТКА_ДОРАХОВАНО|[123]_(ПІБ / Назва|Частка))$").head(10)

## 5. Країна
`громадянства: Україна` → `Україна` (для засновників і бенефіціарів)

In [ ]:
country_cols = [c for c in df.columns if re.match(r"^(Засн|Бенеф)_\d+_Країна$", c)]
for c in country_cols:
    df[c] = df[c].str.replace(r"^.*:\s*", "", regex=True).str.strip()

pd.concat([df[c] for c in country_cols]).replace("", np.nan).value_counts().head(15)

## 6. Збереження

In [ ]:
out = Path(OUTPUT_PATH) if OUTPUT_PATH else INPUT_PATH.with_name(INPUT_PATH.stem + "_processed.csv")
print(df.shape)

df.to_csv(out.with_suffix(".csv"), index=False, encoding="utf-8-sig")
print("CSV  ->", out.with_suffix(".csv"))

if len(df) < 1_048_576 and df.shape[1] <= 16_384:
    try:
        import xlsxwriter  # noqa: F401
        engine = "xlsxwriter"
    except ImportError:
        engine = "openpyxl"
    df.to_excel(out.with_suffix(".xlsx"), index=False, engine=engine)
    print("XLSX ->", out.with_suffix(".xlsx"))